---  
 
## ***Before Beginning***

- Yesterday, you directly wrote the code w = w - lr * w.grad, right? optimizer.step() is like a magical line that does that for you.

#### ***Training Loop***
- The structure of the training loop is used almost as is in the CNN and RNN models you will learn in the future.
- The following pseudo-code is the learning form of most artificial intelligence.

In [ ]:
# 1. Define model, loss function, optimizer
model = MyModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 2. Prepare DataLoader
dataloader = DataLoader(...)

# 3. Training Loop (Repeat N epochs)
for epoch in range(num_epochs):
    # Get mini-batch from DataLoader
    for data, labels in dataloader:
        # 3-1. Initialize Gradients
        optimizer.zero_grad()

        # 3-2. Forward Pass
        outputs = model(data)

        # 3-3. Calculate Loss
        loss = criterion(outputs, labels)

        # 3-4. Backward Pass
        loss.backward()

        # 3-5. Update Parameters
        optimizer.step()

    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

#### ***Common Mistakes***

- Missing optimizer.zero_grad(): "Why do I need to zero the gradients every time?"
- Because PyTorch 'accumulates' gradients instead of 'overwriting' them when calling backward()

- Tensor Device Mismatch (CPU vs GPU):
    * Errors occurring because the model is on the GPU but the data remains on the CPU are very common.
    * You must send the data to the same device as the model within the training loop, like data = data.to(device), labels = labels.to(device).

- Switching the Model's Evaluation Mode:
    * It's not essential, but a brief introduction to the concepts of model.train() and model.eval()
    * It's not very important now, but from Day 5 onwards, there are cases where the model's behavior differs between training and evaluation.
    * It's good to get into the habit of switching between these two modes from now on.  
  
  
---


## ***Day 4 Advanced Supplementary Learning:***
### Mastering Building a Real Neural Network (Revised Edition)

#### Learning Objective: To gain the ability to build a complete training pipeline in a reusable structure by automating repetitive tasks using PyTorch's high-level APIs (nn.Module, Optimizer, DataLoader).

### ***Concept Check Quiz***
This quiz will thoroughly check your understanding of the core concepts.

1. What are the respective roles of the __init__ and forward methods in an nn.Module class?  
    [Answer](Day05_quiz_1.ipynb)  

2. What is the role of an Optimizer, and at which stage of the training loop is it used?  
    [Answer](Day05_quiz_2.ipynb)  

3. What are the respective roles of Dataset and DataLoader, and why should they be used together?  
    [Answer](Day05_quiz_3.ipynb)   

4. List the three core steps of the training loop (optimizer.zero_grad(), loss.backward(), optimizer.step()) in the correct order and explain the role of each step.  
    [Answer](Day05_quiz_4.ipynb)   

5. What does the code nn.Linear(in_features=10, out_features=5) mean? What are the input and output sizes of this layer?  
    [Answer](Day05_quiz_5.ipynb)   

6. What is the term for one full pass through the entire dataset during training? And what does the batch_size in a DataLoader signify?  
    [Answer](Day05_quiz_6.ipynb)   

7. Why do we pass model.parameters() to the optimizer? What would happen if this code were omitted?  
    [Answer](Day05_quiz_7.ipynb)   

8. Name one loss function typically used for regression problems and one for classification problems in PyTorch, and briefly explain their difference.  
    [Answer](Day05_quiz_8.ipynb)   

9. What is the purpose of the loss.item() code? What is the difference if you print the loss tensor itself without .item()?  
    [Answer](Day05_quiz_9.ipynb)   

10. Why do we use model.train() mode for training and model.eval() mode for evaluation? (Hint: Dropout, BatchNorm, etc.)  
    [Answer](Day05_quiz_10.ipynb)   

11. In the code torch.arange(100).view(-1, 1), what role does .view(-1, 1) play?  
    [Answer](Day05_quiz_11.ipynb)   

12. What is the main reason for using activation functions like nn.ReLU in a neural network model? What limitations would a model have without activation functions?  
    [Answer](Day05_quiz_12.ipynb)   

13. While training, the loss value does not decrease at all or even diverges. What hyperparameter should be suspected first, and how should it be adjusted?  
    [Answer](Day05_quiz_13.ipynb)   

14. What are the conceptual differences between the SGD and Adam optimizers? Which one generally tends to converge faster?  
    [Answer](Day05_quiz_14.ipynb)  

15. What should the __getitem__ method of a CustomDataset class return, and in what data type?  
    [Answer](Day05_quiz_15.ipynb)  

16. When loss.backward() is called, what value is stored in the .grad attribute of a tensor that was set with requires_grad=False?  
    [Answer](Day05_quiz_16.ipynb)  

### ***Code Walkthroughs and Exercises***
  

Increase your adaptability to the PyTorch pipeline structure by following code examples for various scenarios and immediately solving related exercises.

- **Topic 1: Basic Linear Regression (Review)**
    - Scenario: Create the most basic regression model that predicts a single output (y) from a single input (x).
    - Core Concepts: nn.Linear(1, 1), nn.MSELoss

In [ ]:
# 0. Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# --- Hyperparameter Settings ---
LEARNING_RATE = 0.0001
EPOCHS = 1000
BATCH_SIZE = 10

# 1. Prepare data: y = 3x + 5
# Training data
X_train = torch.arange(1, 101, 1, dtype=torch.float32).view(-1, 1)
y_train = 3 * X_train + 5 + torch.randn(100, 1) * 2
# [Improvement 1] Test data
X_test = torch.arange(101, 201, 1, dtype=torch.float32).view(-1, 1)
y_test = 3 * X_test + 5 + torch.randn(100, 1) * 2

# 2. Define Dataset and DataLoader
class CustomDataset(Dataset):
    def __init__(self, x, y): self.x, self.y = x, y
    def __getitem__(self, i): return (self.x[i], self.y[i])
    def __len__(self): return len(self.x)

train_dataset = CustomDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 3. Define model, loss function, and optimizer
model = nn.Linear(in_features=1, out_features=1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)

# 4. Training loop
print("--- Starting Linear Regression Training ---")
for epoch in range(EPOCHS):
    for inputs, labels in train_loader:
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {loss.item():.4f}')

# --- [Improvement 1] Model Evaluation ---
model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradient calculation for efficiency
    test_outputs = model(X_test)
    test_loss = criterion(test_outputs, y_test)
    print(f'\n--- Model Evaluation ---\nTest Loss: {test_loss.item():.4f}')

# --- [Improvement 3] Check Learned Parameters ---
print("\n--- Learned Parameters (True: w=3, b=5) ---")
for name, param in model.named_parameters():
    if name == 'weight':
        print(f'Learned Weight: {param.data.item():.4f}')
    elif name == 'bias':
        print(f'Learned Bias: {param.data.item():.4f}')

# --- [Improvement 2] Visualize the Results ---
plt.figure(figsize=(8, 6))
plt.scatter(X_train.numpy(), y_train.numpy(), label='Original Data', alpha=0.6)
predicted = model(X_train).detach().numpy()
plt.plot(X_train.numpy(), predicted, color='r', linewidth=2, label='Fitted Line')
plt.title('Linear Regression Result')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

- **[Topic 1] Programming Exercises**  
    - **Problem 1-1 (Optimizer Swap & Comparison)**: 
        - Change the optimizer in the code above from torch.optim.SGD to torch.optim.Adam and retrain the model. With the same learning rate and number of epochs, which optimizer, SGD or Adam, reduces the loss faster?
    - **Problem 1-2 (Visualize Training Process)**: 
        - Store the calculated loss value for each epoch in a Python list. After training is complete, use Matplotlib to plot a line graph showing the trend of loss reduction against the epochs.  


In [ ]:
# 0. Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

# --- Hyperparameter Settings ---
EPOCHS = 200
BATCH_SIZE = 10

# 1. Prepare data
X_train = torch.arange(1, 101, 1, dtype=torch.float32).view(-1, 1)
y_train = 3 * X_train + 5 + torch.randn(100, 1) * 2

# 2. Define Dataset and DataLoader
class CustomDataset(Dataset):
    def __init__(self, x, y): self.x, self.y = x, y
    def __getitem__(self, i): return (self.x[i], self.y[i])
    def __len__(self): return len(self.x)

train_dataset = CustomDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 3. Define a function to run the comparison
def run_and_plot_comparison(learning_rate):
    """
    Trains models with SGD and Adam using a given learning rate
    and plots the results on two separate subplots within a single figure.
    """
    print(f"\n--- Running Comparison for Learning Rate: {learning_rate} ---")
    
    criterion = nn.MSELoss()

    # --- Training Session 1: SGD Optimizer ---
    print("Training with SGD Optimizer...")
    model_sgd = nn.Linear(in_features=1, out_features=1)
    optimizer_sgd = torch.optim.SGD(model_sgd.parameters(), lr=learning_rate)
    sgd_loss_history = []
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for inputs, labels in train_loader:
            outputs = model_sgd(inputs)
            loss = criterion(outputs, labels)
            optimizer_sgd.zero_grad()
            loss.backward()
            optimizer_sgd.step()
            epoch_loss += loss.item()
        sgd_loss_history.append(epoch_loss / len(train_loader))

    # --- Training Session 2: Adam Optimizer ---
    print("Training with Adam Optimizer...")
    model_adam = nn.Linear(in_features=1, out_features=1)
    optimizer_adam = torch.optim.Adam(model_adam.parameters(), lr=learning_rate)
    adam_loss_history = []
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for inputs, labels in train_loader:
            outputs = model_adam(inputs)
            loss = criterion(outputs, labels)
            optimizer_adam.zero_grad()
            loss.backward()
            optimizer_adam.step()
            epoch_loss += loss.item()
        adam_loss_history.append(epoch_loss / len(train_loader))

    # --- Visualize the Results on Subplots ---
    # Create a figure and a set of subplots (2 rows, 1 column)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12), sharex=True)
    
    # Main title for the entire figure
    fig.suptitle(f'Optimizer Comparison (LR = {learning_rate})', fontsize=16)

    # Plot 1: SGD Loss on the first subplot (ax1)
    ax1.plot(range(EPOCHS), sgd_loss_history, label='SGD Loss', color='blue')
    ax1.set_title('SGD Loss Over Time')
    ax1.set_ylabel('Loss')
    ax1.grid(True)
    ax1.legend()

    # Plot 2: Adam Loss on the second subplot (ax2)
    ax2.plot(range(EPOCHS), adam_loss_history, label='Adam Loss', color='red')
    ax2.set_title('Adam Loss Over Time')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.grid(True)
    ax2.legend()
    
    # Adjust layout to prevent titles from overlapping
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.show()


# --- Run the experiments for both learning rates ---
run_and_plot_comparison(learning_rate=0.0003)
run_and_plot_comparison(learning_rate=0.001)

---  

- **Topic 2: Multiple Linear Regression**
    - Scenario: Create a model that predicts a single output (y) from two inputs (x1, x2).
    - Core Concept: nn.Linear(2, 1). This handles the situation where the number of input features increases from one to two.


In [ ]:
# [Topic 2] Core Code
# 1. Prepare data: y = 2*x1 + 3*x2 + 4
X2_1 = torch.randn(200, 1)
X2_2 = torch.randn(200, 1)
X2 = torch.cat([X2_1, X2_2], dim=1) # Combine the two input features
y2 = 2 * X2_1 + 3 * X2_2 + 4 + torch.randn(200, 1) * 2

# 2. Define Dataset and DataLoader (reusing CustomDataset)
train_dataset2 = CustomDataset(X2, y2)
train_loader2 = DataLoader(train_dataset2, batch_size=10, shuffle=True)

# 3. Define model, loss function, and optimizer
# Since there are 2 input features, set in_features=2
model2 = nn.Linear(in_features=2, out_features=1)
criterion2 = nn.MSELoss()
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.01)

# 4. Training loop
print("\n--- Starting Multiple Linear Regression Training ---")
for epoch in range(100):
    for inputs, labels in train_loader2:
        outputs = model2(inputs)
        loss = criterion2(outputs, labels)
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')


- **[Topic 2] Programming Exercises**
    - **Problem 2-1 (Model Architecture Change)**: 
        - The current model is a single linear layer. Create a deeper model by adding a hidden layer, like nn.Linear(in_features=2, out_features=10) -> nn.ReLU() -> nn.Linear(in_features=10, out_features=1). Confirm that the training loop code remains almost the same even when the model's architecture changes.

    - **Problem 2-2 (Learning Rate Tuning)**: 
        - In the optimizer from the code above, change the learning rate (lr) to 10 times (0.1) and 0.1 times (0.001) its current value and run the training again. Observe how the loss value changes (e.g., diverges or decreases too slowly) when the learning rate is too high or too low, and think about the reason.  
---

- **Topic 3: Simple Binary Classification**
    - **Scenario**: Create a model that classifies a score (x) as pass (1) if it's 50 or above, and fail (0) otherwise.
    - **Core Concepts**: nn.Sigmoid activation function, nn.BCELoss (Binary Cross Entropy Loss)

In [ ]:
# [Topic 3] Core Code
# 1. Prepare data: Pass/Fail based on a score of 50
X3 = torch.arange(0, 100, 1, dtype=torch.float32).view(-1, 1)
# 1 (Pass) if score is >= 50, otherwise 0 (Fail)
y3 = (X3 >= 50).float()

# 2. Define Dataset and DataLoader (reusing CustomDataset)
train_dataset3 = CustomDataset(X3, y3)
train_loader3 = DataLoader(train_dataset3, batch_size=10, shuffle=True)

# 3. Define model, loss function, and optimizer
class BinaryClassifier(nn.Module):
    def __init__(self):
        super(BinaryClassifier, self).__init__()
        self.linear = nn.Linear(1, 1)
        # Sigmoid function to convert output to a probability between 0 and 1
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.linear(x))

model3 = BinaryClassifier()
# For binary classification problems, use BCELoss
criterion3 = nn.BCELoss()
optimizer3 = torch.optim.Adam(model3.parameters(), lr=0.1)

# 4. Training loop
print("\n--- Starting Binary Classification Training ---")
for epoch in range(100):
    for inputs, labels in train_loader3:
        outputs = model3(inputs)
        loss = criterion3(outputs, labels)
        optimizer3.zero_grad()
        loss.backward()
        optimizer3.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')

# 5. Check results after training
print("\n--- Testing Classification Results ---")
model3.eval() # Switch to evaluation mode
with torch.no_grad(): # Disable gradient calculation
    test_scores = torch.tensor([[10.0], [49.0], [51.0], [95.0]])
    predictions = model3(test_scores)
    # If predicted probability is >= 0.5, it's a Pass (True)
    results = predictions >= 0.5
    for score, result in zip(test_scores, results):
        print(f"Score: {score.item():.0f} -> Pass: {result.item()}")


- **[Topic 3] Programming Exercises**
    - **Problem 3-1 (Calculate Accuracy)**: 
        - After training is complete, feed the entire training dataset (X3, y3) into the model to make predictions. Compare the predictions with the actual labels to calculate the model's accuracy. (Accuracy = Number of Correct Predictions / Total Number of Predictions)

    - **Problem 3-2 (Save and Load Model)**: 
        - Save the parameters of the trained BinaryClassifier model to a file using torch.save(model3.state_dict(), 'classifier.pth'). Then, create a new model object, load the saved parameters using model.load_state_dict(torch.load('classifier.pth')), and confirm that the model is in its trained state by re-running the "Testing Classification Results" code.  
--- 

### ***3. Mini-Projects***

Choose two of the following three topics and build a model from scratch using a real dataset.

#### ***Project A***: California Housing Price Prediction (Regression)
- Objective: 
    - Use the California housing dataset from Scikit-learn to create a regression model that predicts housing prices based on various housing-related features (median income, house age, etc.).

- Data: sklearn.datasets.fetch_california_housing

- Core Task: Build a multiple linear regression model that takes 8 input features to predict 1 housing price, and train it using MSELoss.


In [ ]:
# 0. Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Import from Scikit-learn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Hyperparameter Settings ---
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.01

# 1. Load and Prepare Data
# Load the dataset from Scikit-learn
housing = fetch_california_housing()
X, y = housing.data, housing.target

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features: This is crucial for stable training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert NumPy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# 2. Define Custom Dataset and DataLoader
class HousingDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = HousingDataset(X_train_tensor, y_train_tensor)
test_dataset = HousingDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 3. Define the Multiple Linear Regression Model
class RegressionModel(nn.Module):
    def __init__(self, num_features):
        super(RegressionModel, self).__init__()
        # A single linear layer is sufficient for multiple linear regression
        self.linear = nn.Linear(num_features, 1)

    def forward(self, x):
        return self.linear(x)

# The dataset has 8 features
model = RegressionModel(num_features=8)

# 4. Define Loss Function and Optimizer
criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 5. Training and Evaluation Loop
train_losses = []
test_losses = []

print("--- Starting Model Training ---")
for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    running_train_loss = 0.0
    for features, labels in train_loader:
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
    
    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)
    
    # --- Evaluation ---
    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        for features, labels in test_loader:
            outputs = model(features)
            loss = criterion(outputs, labels)
            running_test_loss += loss.item()
            
    epoch_test_loss = running_test_loss / len(test_loader)
    test_losses.append(epoch_test_loss)
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {epoch_train_loss:.4f}, Test Loss: {epoch_test_loss:.4f}')

print("--- Training Finished ---")

# 6. Visualize the Results

# Plot 1: Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Training and Test Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# Plot 2: Predictions vs. Actuals
plt.figure(figsize=(8, 8))
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor).detach().numpy()
plt.scatter(y_test, predictions, alpha=0.3)
# Add a diagonal line for reference (perfect prediction)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')
plt.title('Actual vs. Predicted Housing Prices')
plt.xlabel('Actual Prices')
plt.ylabel('Predicted Prices')
plt.grid(True)
plt.show()

#### ***Project B***: Breast Cancer Diagnosis (Binary Classification)
- Objective: 
    - Use the breast cancer dataset from Scikit-learn to create a model that classifies tumors as malignant or benign based on various tumor characteristics.

- Data: sklearn.datasets.load_breast_cancer

- Core Task: Build a binary classification model that takes 30 input features to predict 1 classification result (0 or 1). You must use Sigmoid and BCELoss.

In [ ]:
# 0. Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Import from Scikit-learn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# --- Hyperparameter Settings ---
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# 1. Load and Prepare Data
# Load the dataset
cancer_data = load_breast_cancer()
X, y = cancer_data.data, cancer_data.target

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features for better performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert NumPy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1) # Reshape for BCELoss
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1) # Reshape for BCELoss

# 2. Define Custom Dataset and DataLoader
class CancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = CancerDataset(X_train_tensor, y_train_tensor)
test_dataset = CancerDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 3. Define the Binary Classification Model
class ClassificationModel(nn.Module):
    def __init__(self, num_features):
        super(ClassificationModel, self).__init__()
        # A simple model with one linear layer
        self.linear = nn.Linear(num_features, 1)
    
    def forward(self, x):
        # Pass through the linear layer and then a Sigmoid activation function
        # to get a probability output between 0 and 1.
        return torch.sigmoid(self.linear(x))

# The dataset has 30 features
model = ClassificationModel(num_features=30)

# 4. Define Loss Function and Optimizer
# Binary Cross-Entropy Loss is used for binary classification problems
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 5. Training and Evaluation Loop
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

print("--- Starting Model Training ---")
for epoch in range(EPOCHS):
    model.train()
    
    # --- Training ---
    for features, labels in train_loader:
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    # --- Evaluation on both training and test set ---
    model.eval()
    with torch.no_grad():
        # Calculate training loss and accuracy
        train_outputs = model(X_train_tensor)
        train_loss = criterion(train_outputs, y_train_tensor)
        train_losses.append(train_loss.item())
        predicted_train = (train_outputs >= 0.5).float()
        train_acc = accuracy_score(y_train_tensor.numpy(), predicted_train.numpy())
        train_accuracies.append(train_acc)
        
        # Calculate test loss and accuracy
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_losses.append(test_loss.item())
        predicted_test = (test_outputs >= 0.5).float()
        test_acc = accuracy_score(y_test_tensor.numpy(), predicted_test.numpy())
        test_accuracies.append(test_acc)
        
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_acc:.4f}')

print("--- Training Finished ---")

# 6. Visualize the Results

# Plot 1: Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (BCE)')
plt.legend()
plt.grid(True)
plt.show()

# Plot 2: Accuracy Curve
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(test_accuracies, label='Test Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Plot 3: Confusion Matrix
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred_class = (y_pred >= 0.5).float().numpy()
cm = confusion_matrix(y_test, y_pred_class)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cancer_data.target_names)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

#### ***Project C***: Wine Type Classification (Multi-class Classification - Challenge!)
- Objective: 
    - Use the wine dataset from Scikit-learn to create a model that classifies wine into one of three types based on its chemical properties.

- Data: sklearn.datasets.load_wine

- Core Task: Build a model that takes 13 input features to classify into 3 classes. The out_features of the final output layer should be 3, and you should use nn.CrossEntropyLoss as the loss function. (Hint: In multi-class classification, Softmax is used instead of Sigmoid at the end, but nn.CrossEntropyLoss includes Softmax internally, so your model's forward method only needs to return the result of the final nn.Linear layer.)

In [ ]:
# 0. Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Import from Scikit-learn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# --- Hyperparameter Settings ---
EPOCHS = 150
BATCH_SIZE = 16
LEARNING_RATE = 0.005
NUM_FEATURES = 13
NUM_CLASSES = 3

# 1. Load and Prepare Data
# Load the dataset
wine_data = load_wine()
X, y = wine_data.data, wine_data.target

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert NumPy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
# For CrossEntropyLoss, the target must be of type LongTensor (int64) and not be one-hot encoded
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 2. Define Custom Dataset and DataLoader
class WineDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = WineDataset(X_train_tensor, y_train_tensor)
test_dataset = WineDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 3. Define the Multi-class Classification Model
class MultiClassModel(nn.Module):
    def __init__(self, num_features, num_classes):
        super(MultiClassModel, self).__init__()
        # A simple Multi-Layer Perceptron (MLP)
        self.layer_1 = nn.Linear(num_features, 64)
        self.layer_2 = nn.Linear(64, 32)
        self.layer_out = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        # The raw output scores (logits) are returned.
        # No Softmax is needed here because nn.CrossEntropyLoss applies it internally.
        x = self.layer_out(x)
        return x

model = MultiClassModel(num_features=NUM_FEATURES, num_classes=NUM_CLASSES)

# 4. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 5. Training and Evaluation Loop
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

print("--- Starting Model Training ---")
for epoch in range(EPOCHS):
    model.train()
    
    # --- Training ---
    for features, labels in train_loader:
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    # --- Evaluation ---
    model.eval()
    with torch.no_grad():
        # Calculate training loss and accuracy
        train_outputs = model(X_train_tensor)
        train_loss = criterion(train_outputs, y_train_tensor)
        train_losses.append(train_loss.item())
        # Get predictions by finding the class with the highest score
        predicted_train = torch.argmax(train_outputs, dim=1)
        train_acc = accuracy_score(y_train_tensor.numpy(), predicted_train.numpy())
        train_accuracies.append(train_acc)
        
        # Calculate test loss and accuracy
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_losses.append(test_loss.item())
        predicted_test = torch.argmax(test_outputs, dim=1)
        test_acc = accuracy_score(y_test_tensor.numpy(), predicted_test.numpy())
        test_accuracies.append(test_acc)
        
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_acc:.4f}')

print("--- Training Finished ---")
print(f"Final Test Accuracy: {test_accuracies[-1]:.4f}")

# 6. Visualize the Results

# Plot 1: Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.legend()
plt.grid(True)
plt.show()

# Plot 2: Accuracy Curve
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(test_accuracies, label='Test Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Plot 3: Confusion Matrix
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred_class = torch.argmax(y_pred, dim=1).numpy()
cm = confusion_matrix(y_test, y_pred_class)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=wine_data.target_names)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()